# 02 · Choix du site et des 20 bâtiments

**Contexte** : on ne le décide pas à l'avance. Il se **déduit des bâtiments retenus et de leur façon de consommer** (campus universitaire, parc de bureaux, collectivité, parc tertiaire…). Ce notebook choisit le site et les bâtiments ; le notebook 04 (section 3 bis) analyse leur consommation et propose le contexte qui en découle.

**Démarche** : on mesure d'abord la qualité de **tous** les bâtiments, on choisit le **site** d'après des critères écrits, puis on filtre et on tire **20 bâtiments** avec une graine fixe (reproductible), en gardant la diversité d'usages.

**Ce qu'on produit** (dans `data/interim/`, hors Git) : `batiments-retenus.csv`, `entonnoir.csv`, `selection-params.json`. La décision et ses justifications seront recopiées dans l'ADR « choix du jeu de données » ; à la séance 3, la liste des bâtiments et les seuils iront dans `monprojet/config.py`.

> Règle d'or : on écrit les critères, on regarde les chiffres, **puis** on fixe les seuils. Pas l'inverse.

In [8]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("..")                       # ce notebook s'exécute depuis notebooks/
RAW = ROOT / "data" / "raw" / "bdg2"    # fichiers téléchargés (jamais modifiés, jamais dans Git)
INTERIM = ROOT / "data" / "interim"     # résultats intermédiaires (hors Git)
PROCESSED = ROOT / "data" / "processed" # jeu final (hors Git, DVC en S7)
for dossier in (INTERIM, PROCESSED):
    dossier.mkdir(parents=True, exist_ok=True)

SEED = 42
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.2f}".format)
plt.rcParams["figure.dpi"] = 100

In [9]:
def runs(masque):
    """Pour chaque heure où masque est True : longueur de la série de True qui la contient (0 sinon).
    Exemple : [F, T, T, T, F, T] -> [0, 3, 3, 3, 0, 1]"""
    groupe = (masque != masque.shift()).cumsum()
    return masque.groupby(groupe).transform("sum") * masque

def dominant(serie):
    """Valeur la plus fréquente d'une colonne (ou 'inconnu' si tout est vide)."""
    vc = serie.value_counts()
    return vc.index[0] if len(vc) else "inconnu"

def plus_long_trou(serie):
    """Plus longue suite d'heures manquantes consécutives."""
    return int(runs(serie.isna()).max())

In [10]:
meta = pd.read_csv(RAW / "metadata.csv")
elec = pd.read_csv(RAW / "electricity.csv", parse_dates=["timestamp"], index_col="timestamp")
weather = pd.read_csv(RAW / "weather.csv", parse_dates=["timestamp"])
print(meta.shape, elec.shape, weather.shape)

(1636, 32) (17544, 1578) (331166, 10)


## 1. Qualité de chaque bâtiment (données raw)

Pour chaque colonne d'électricité on mesure : % de valeurs manquantes, plus long trou (en heures), nombre de zéros et de valeurs négatives, consommation moyenne, consommation par m². Ces chiffres serviront à fixer les seuils.

In [11]:
colonnes = elec.columns.intersection(meta["building_id"])
m = meta.drop_duplicates("building_id").set_index("building_id").loc[colonnes].copy()
m.index.name = "building_id"
m["pct_nan"] = elec[colonnes].isna().mean() * 100
m["plus_long_trou_h"] = elec[colonnes].apply(plus_long_trou)
m["pct_zeros"] = (elec[colonnes] == 0).mean() * 100
m["nb_negatifs"] = (elec[colonnes] < 0).sum()
m["kwh_moyen"] = elec[colonnes].mean()
m["kwh_par_m2"] = m["kwh_moyen"] / m["sqm"]
print(len(m), "bâtiments avec une colonne d'électricité ET une ligne de métadonnées")
m[["site_id", "primaryspaceusage", "sqm", "pct_nan", "plus_long_trou_h", "pct_zeros", "nb_negatifs", "kwh_par_m2"]].describe().T

1578 bâtiments avec une colonne d'électricité ET une ligne de métadonnées


,count,mean,std,min,25%,50%,75%,max
sqm,1578.00,8345.47,10102.88,26.30,2046.22,5261.35,10424.83,81290.10
pct_nan,1578.00,4.74,13.75,0.00,0.01,0.15,0.95,100.00
plus_long_trou_h,1578.00,701.44,2283.05,0.00,1.00,6.00,48.00,17544.00
pct_zeros,1578.00,4.19,11.62,0.00,0.00,0.03,1.70,100.00
nb_negatifs,1578.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
kwh_par_m2,1576.00,0.02,0.03,0.00,0.01,0.01,0.02,0.49


## 2. Comparer les sites

Un site = une météo, un fuseau horaire, un pays (donc les mêmes jours fériés). On le choisit sur des critères **écrits avant** de regarder le tableau.

**Critères de site proposés** (les seuils se fixent après lecture du tableau) :
1. assez de bâtiments pour avoir un vrai choix de 20 (au moins ~30 candidats après filtres) ;
2. plusieurs usages représentés, cohérents avec un campus ;
3. météo du site complète : toutes les heures présentes (`meteo_heures_manquantes` = 0) et peu de valeurs manquantes pour la température, et si possible pour le vent et la pression (`cloudCoverage` est très vide dans l'ensemble : à lire site par site) ;
4. un seul fuseau horaire ;
5. électricité raw de qualité acceptable (médiane de NaN et de zéros).

In [12]:
meteo_unique = weather.drop_duplicates(["site_id", "timestamp"])
meteo_site = meteo_unique.groupby("site_id").agg(
    meteo_heures=("timestamp", "size"),
    meteo_nan_temp=("airTemperature", lambda s: s.isna().mean() * 100),
    meteo_nan_nuages=("cloudCoverage", lambda s: s.isna().mean() * 100),
    meteo_nan_pluie1h=("precipDepth1HR", lambda s: s.isna().mean() * 100),
)
meteo_site["meteo_heures_manquantes"] = 17544 - meteo_site["meteo_heures"]
par_site = m.groupby("site_id").agg(
    nb_batiments=("sqm", "size"),
    nb_usages=("primaryspaceusage", "nunique"),
    usage_dominant=("primaryspaceusage", dominant),
    usage_vide=("primaryspaceusage", lambda s: int(s.isna().sum())),
    nan_median=("pct_nan", "median"),
    zeros_median=("pct_zeros", "median"),
    nb_fuseaux=("timezone", "nunique"),
).join(meteo_site)
par_site.sort_values("nb_batiments", ascending=False)

,nb_batiments,nb_usages,usage_dominant,usage_vide,nan_median,zeros_median,nb_fuseaux,meteo_heures,meteo_nan_temp,meteo_nan_nuages,meteo_nan_pluie1h,meteo_heures_manquantes
site_id,,,,,,,,,,,,
Rat,305,11,Education,0,0.43,0.01,1,17539,0.03,42.74,0.28,5
Hog,152,13,Office,0,0.00,0.00,1,17542,0.01,49.24,0.29,2
Lamb,146,10,Education,0,0.00,0.59,1,17500,0.01,68.68,100.00,44
Fox,137,12,Education,0,0.01,0.03,1,17543,0.01,28.33,0.40,1
Bull,123,6,Education,0,0.15,1.72,1,17529,0.19,39.51,0.11,15
Cockatoo,117,9,Education,0,8.15,0.00,1,16975,0.01,54.10,78.76,569
Eagle,106,7,Office,0,0.00,1.08,1,17536,0.02,39.96,0.55,8
Panther,105,7,Education,0,0.07,19.25,1,17544,0.02,42.99,0.05,0
Bear,92,7,Education,0,0.24,0.00,1,17542,0.00,46.43,8.39,2


In [13]:
# Répartition des usages par site (les 8 usages les plus fréquents)
top = m["primaryspaceusage"].value_counts().index[:8]
pd.crosstab(m["site_id"], m["primaryspaceusage"])[top]

primaryspaceusage,Education,Office,Entertainment/public assembly,Public services,Lodging/residential,Healthcare,Other,Parking
site_id,,,,,,,,
Bear,67,0,9,6,4,0,0,3
Bobcat,17,5,4,1,3,0,3,0
Bull,63,17,17,2,19,0,0,0
Cockatoo,43,18,16,6,28,0,0,0
Crow,5,0,0,0,0,0,0,0
Eagle,26,40,11,7,10,10,0,0
Fox,63,24,21,6,12,1,0,3
Gator,0,8,26,29,0,0,9,0
Hog,24,73,11,5,11,3,3,8


## 3. Décision : le site

**Pré-lecture de `metadata.csv` (constats réels)** : un site dont tous les usages sont vides ne sert à rien (colonne `usage_vide` = `nb_batiments`) ; un site de moins de ~30 bâtiments ne permet pas un vrai choix de 20 ; les 4 sites sans coordonnées GPS ne sont pas un obstacle ici (le fuseau `timezone` suffit).

**À FAIRE À LA MAIN** : lis les deux tableaux, applique tes critères, puis remplis la cellule suivante et **écris la justification** (elle ira dans l'ADR) (chiffres à l'appui, sites écartés et pourquoi).

In [14]:
SITE = None   # <-- À REMPLIR, par exemple SITE = "NomDuSite"
assert SITE is not None, "Choisis un site dans le tableau ci-dessus puis remplace None par son nom."
assert SITE in par_site.index, f"Site inconnu. Sites possibles : {par_site.index.tolist()}"
site_m = m[m["site_id"] == SITE]
print(f"Site {SITE} : {len(site_m)} bâtiments avec électricité")
site_m["primaryspaceusage"].value_counts()

AssertionError: Choisis un site dans le tableau ci-dessus puis remplace None par son nom.

## 4. Filtres sur les bâtiments du site

Les seuils ci-dessous sont des **valeurs de départ** : regarde les graphiques de la cellule suivante, ajuste, et note chaque choix (il ira dans l'ADR) avec le chiffre qui le justifie.

In [ ]:
# --- Seuils (à ajuster après lecture des graphiques) ---
MAX_NAN = 10.0          # % de valeurs manquantes max sur 2 ans
MAX_TROU_H = 24 * 14    # plus long trou toléré (heures)
MAX_ZEROS = 10.0        # % de zéros max
USAGES_EXCLUS = []      # usages à écarter, ex. ["Other"]
MIN_PAR_USAGE = 2       # nb minimal de bâtiments par usage dans le tirage
MIN_CANDIDATS_USAGE = 5 # un usage représenté par moins de candidats est écarté (trop rare pour être appris)
N_BATIMENTS = 20

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
for a, (col, seuil, titre) in zip(ax, [("pct_nan", MAX_NAN, "% de NaN"), ("plus_long_trou_h", MAX_TROU_H, "Plus long trou (h)"), ("pct_zeros", MAX_ZEROS, "% de zéros")]):
    valeurs = site_m[col].sort_values().to_numpy()
    a.plot(valeurs, np.arange(1, len(valeurs) + 1) / len(valeurs))
    a.axvline(seuil, color="red", ls="--", label=f"seuil = {seuil}")
    a.set_title(titre + " : part des bâtiments sous le seuil"); a.legend()
plt.tight_layout()

In [ ]:
etapes = [("Bâtiments du site avec électricité", site_m)]
c = site_m[site_m["primaryspaceusage"].notna()];  etapes.append(("Usage renseigné", c))
c = c[~c["primaryspaceusage"].isin(USAGES_EXCLUS)]; etapes.append((f"Usages exclus : {USAGES_EXCLUS}", c))
c = c[c["pct_nan"] <= MAX_NAN];            etapes.append((f"% NaN <= {MAX_NAN}", c))
c = c[c["plus_long_trou_h"] <= MAX_TROU_H]; etapes.append((f"Plus long trou <= {MAX_TROU_H} h", c))
c = c[c["pct_zeros"] <= MAX_ZEROS];        etapes.append((f"% zéros <= {MAX_ZEROS}", c))
c = c[(c["kwh_moyen"] > 0) & c["sqm"].notna() & (c["sqm"] > 0)]; etapes.append(("Consommation > 0 et surface connue", c))
effectifs = c["primaryspaceusage"].value_counts()
c = c[c["primaryspaceusage"].map(effectifs) >= MIN_CANDIDATS_USAGE]; etapes.append((f"Usage avec >= {MIN_CANDIDATS_USAGE} candidats", c))
candidats = c
entonnoir = pd.DataFrame([(nom, len(d)) for nom, d in etapes], columns=["etape", "nb_batiments"])
entonnoir

In [ ]:
assert len(candidats) >= N_BATIMENTS, f"Seulement {len(candidats)} candidats pour {N_BATIMENTS} bâtiments : assouplis un seuil ou change de site."
print(candidats["primaryspaceusage"].value_counts())

## 5. Tirage de 20 bâtiments (stratifié, reproductible)

**Pourquoi pas « les 20 plus complets »** : on choisirait toujours les bâtiments faciles, et le modèle serait testé sur un échantillon trop favorable.
**Pourquoi stratifié** : on garantit quelques bâtiments de chaque usage, comme sur un vrai campus, puis on complète au hasard. La graine fixe (42) rend le tirage identique pour tout le monde.

In [ ]:
def tirage_stratifie(cand, n, min_par_usage, graine):
    rng = np.random.default_rng(graine)
    choisis = []
    for usage, groupe in cand.groupby("primaryspaceusage"):
        k = min(min_par_usage, len(groupe))
        choisis += rng.choice(groupe.index.to_numpy(), size=k, replace=False).tolist()
    assert len(choisis) <= n, f"{len(choisis)} bâtiments réservés pour {n} : baisse MIN_PAR_USAGE ou exclus des usages rares."
    reste = cand.index.difference(choisis).to_numpy()
    choisis += rng.choice(reste, size=n - len(choisis), replace=False).tolist()
    return sorted(choisis)

choisis = tirage_stratifie(candidats, N_BATIMENTS, MIN_PAR_USAGE, SEED)
retenus = candidats.loc[choisis]
assert len(retenus) == N_BATIMENTS and retenus.index.is_unique
print(len(retenus), "bâtiments retenus")
pd.concat({"candidats": candidats["primaryspaceusage"].value_counts(),
           "retenus": retenus["primaryspaceusage"].value_counts()}, axis=1).fillna(0).astype(int)

In [ ]:
# L'échantillon ressemble-t-il à la population des candidats ? (surface et conso par m²)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for a, col in zip(ax, ["sqm", "kwh_par_m2"]):
    a.hist(candidats[col], bins=20, alpha=0.5, density=True, label="candidats")
    a.hist(retenus[col], bins=10, alpha=0.5, density=True, label="retenus")
    a.set_title(col); a.legend()
plt.tight_layout()
retenus[["sqm", "kwh_par_m2", "pct_nan", "plus_long_trou_h"]].describe().T

## 6. Sauvegarde des résultats de la sélection (dans `data/interim/`)

In [ ]:
retenus.reset_index()[["building_id", "site_id", "primaryspaceusage", "sqm"]].to_csv(INTERIM / "batiments-retenus.csv", index=False)
entonnoir.to_csv(INTERIM / "entonnoir.csv", index=False)
params = {"site": SITE, "n_batiments": N_BATIMENTS, "graine": SEED, "max_nan_pct": MAX_NAN,
          "max_trou_h": MAX_TROU_H, "max_zeros_pct": MAX_ZEROS, "usages_exclus": USAGES_EXCLUS,
          "min_par_usage": MIN_PAR_USAGE, "min_candidats_usage": MIN_CANDIDATS_USAGE, "nb_candidats": int(len(candidats))}
(INTERIM / "selection-params.json").write_text(json.dumps(params, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(params, indent=2, ensure_ascii=False))

## 7. (Facultatif) Petit échantillon à partager

Les fichiers BDG2 complets sont lourds (≈ 370 Mo). Cette cellule crée un **petit zip** (quelques Mo) avec seulement les 20 bâtiments retenus (raw et cleaned), la météo du site et leurs métadonnées. Utile pour faire relire le travail, ou pour que les camarades démarrent avant que DVC soit en place. Les données BDG2 sont sous licence CC BY-SA 4.0 : cite la source si tu les diffuses. Le zip reste dans `data/interim/` (jamais dans Git).

In [ ]:
import zipfile
ids_retenus = retenus.index.tolist()
fichiers = []
for nom in ("electricity", "electricity_cleaned"):
    sortie = INTERIM / f"{nom}_20.csv"
    pd.read_csv(RAW / f"{nom}.csv", usecols=["timestamp"] + ids_retenus).to_csv(sortie, index=False)
    fichiers.append(sortie)
for nom, df_ in [("weather_site.csv", weather[weather["site_id"] == SITE]), ("metadata_20.csv", meta[meta["building_id"].isin(ids_retenus)])]:
    df_.to_csv(INTERIM / nom, index=False); fichiers.append(INTERIM / nom)
with zipfile.ZipFile(INTERIM / "echantillon-20-batiments.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in fichiers:
        z.write(f, f.name)
print("Zip :", round((INTERIM / "echantillon-20-batiments.zip").stat().st_size / 1e6, 1), "Mo")

## 8. À recopier dans l'ADR « choix du jeu de données »

| Question | Réponse |
|---|---|
| Site retenu et pourquoi (chiffres) | [...] |
| Sites écartés et pourquoi | [...] |
| Seuils retenus et chiffre qui justifie chacun | NaN [...] · trou [...] · zéros [...] |
| Entonnoir | voir `data/interim/entonnoir.csv` |
| Diversité d'usages des 20 bâtiments | [...] |
| L'échantillon est-il représentatif des candidats ? | [...] |